In [4]:
import json
from pathlib import Path

In [5]:
"""
clean_data.py

Cleans the Yelp business dataset to extract only Indianapolis restaurants and writes them
out as newline-delimited JSON with just `business_id` and `name`.
"""

import json
from pathlib import Path

def clean_restaurants():
    """
    Reads the Yelp business JSONL file, filters for restaurants in Indianapolis,
    and writes a cleaned JSONL containing only `business_id` and `name`.

    Inputs:
    - ~/Downloads/yelp_academic_dataset_business.json  (original Yelp dataset)
    - ~/Downloads/cleaned_indianapolis_restaurants.json (output file)

    Outputs:
    - A newline-delimited JSON file where each line is:
        {"business_id": "...", "name": "..."}
    - Prints the total number written and shows the first 5 entries for verification.
    """
    # 1. Define paths to the downloaded dataset and the cleaned output
    DL      = Path.home() / "Downloads"
    INPUT   = DL / "yelp_academic_dataset_business.json"            # raw Yelp business data
    OUTPUT  = DL / "cleaned_indianapolis_restaurants.json"          # filtered output

    head = []    # will hold the first 5 cleaned records for preview
    count = 0    # counter for total restaurants written

    # 2. Open the input file for reading and output file for writing
    with INPUT.open("r", encoding="utf-8") as fin, OUTPUT.open("w", encoding="utf-8") as fout:
        for line in fin:
            rec = json.loads(line)            # parse one JSON record
            city = rec.get("city", "")        # extract city (default empty)
            cats = rec.get("categories", "") or ""  # extract categories string (or blank)

            # 3. Filter: only Indianapolis restaurants (case-insensitive match)
            if city.lower() == "indianapolis" and "restaurant" in cats.lower():
                # 4. Construct the minimal output record
                out = {
                    "business_id": rec["business_id"],
                    "name":        rec["name"],
                }
                # 5. Write the cleaned record as JSONL
                fout.write(json.dumps(out) + "\n")

                # 6. Collect up to 5 examples for preview
                if len(head) < 5:
                    head.append(out)

                count += 1  # increment the total count

    # 7. Print a summary and preview to the console
    print(f"Wrote {count} restaurants to {OUTPUT}\n")
    print("First 5 cleaned restaurants:")
    for r in head:
        print(r)

if __name__ == "__main__":
    clean_restaurants()

Wrote 2862 restaurants to /Users/SHANAYA/Downloads/cleaned_indianapolis_restaurants.json

First 5 cleaned restaurants:
{'business_id': 'il_Ro8jwPlHresjw9EGmBg', 'name': "Denny's"}
{'business_id': 'kfNv-JZpuN6TVNSO6hHdkw', 'name': 'Hibachi Express'}
{'business_id': 'seKihQKpGGnCeLuELRQPSQ', 'name': 'Twin Peaks'}
{'business_id': 'L_TT0BFmFwORAMaA83K54A', 'name': 'Village Tap Room'}
{'business_id': 'tSFXJ0GFl5iUdy021YgWLw', 'name': 'The Mad Griddle '}


In [6]:
"""
clean_reviews.py

Reads the full Yelp review JSONL file, filters for reviews of Indianapolis restaurants
(using the cleaned restaurant list), and writes a minimal JSONL containing only user_id,
business_id, and star rating.
"""

import json
from pathlib import Path

def clean_reviews():
    """
    Filters reviews to those belonging to Indianapolis restaurants.

    Inputs:
    - ~/Downloads/yelp_academic_dataset_review.json         (raw Yelp reviews)
    - ~/Downloads/cleaned_indianapolis_restaurants.json    (filtered restaurant IDs)

    Outputs:
    - ~/Downloads/cleaned_indianapolis_reviews.json
      newline-delimited JSON with fields: user_id, business_id, stars
    - Prints total count and first 5 cleaned reviews for verification.
    """
    # 1. Define directories and file paths
    DL       = Path.home() / "Downloads"
    INPUT    = DL / "yelp_academic_dataset_review.json"          # raw Yelp review data
    BUS_FILE = DL / "cleaned_indianapolis_restaurants.json"      # list of valid restaurant IDs
    OUTPUT   = DL / "cleaned_indianapolis_reviews.json"          # filtered output

    # 2. Load valid Indianapolis business IDs into a set (lowercased for consistency)
    rest_ids = set()
    with BUS_FILE.open("r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            # Add each business_id to the set for fast membership tests
            rest_ids.add(rec["business_id"].lower())

    head = []  # store first 5 outputs for a preview
    count = 0  # total reviews written

    # 3. Read raw reviews line-by-line, filter, and write minimal JSONL
    with INPUT.open("r", encoding="utf-8") as fin, OUTPUT.open("w", encoding="utf-8") as fout:
        for line in fin:
            rec = json.loads(line)  # parse the review record
            biz  = rec["business_id"].lower()
            # 4. Only keep reviews whose business_id is in our Indianapolis set
            if biz in rest_ids:
                # 5. Build a minimal record
                out = {
                    "user_id":     rec["user_id"],
                    "business_id": rec["business_id"],
                    "stars":       rec["stars"],
                }
                # 6. Write one JSON line
                fout.write(json.dumps(out) + "\n")
                # 7. Collect preview items
                if len(head) < 5:
                    head.append(out)
                count += 1

    # 8. Print summary & preview
    print(f"Wrote {count} reviews to {OUTPUT}\n")
    print("First 5 cleaned reviews:")
    for r in head:
        print(r)

if __name__ == "__main__":
    clean_reviews()

Wrote 250579 reviews to /Users/SHANAYA/Downloads/cleaned_indianapolis_reviews.json

First 5 cleaned reviews:
{'user_id': 'ZbqSHbgCjzVAqaa7NKWn5A', 'business_id': 'EQ-TZ2eeD_E0BHuvoaeG5Q', 'stars': 4.0}
{'user_id': 'RreNy--tOmXMl1en0wiBOg', 'business_id': 'cPepkJeRMtHapc_b2Oe_dw', 'stars': 4.0}
{'user_id': 'DBYhpb5hrAYgQjQaMhNYyQ', 'business_id': 'oJ4ik-4PZe6gexxW-tSmsw', 'stars': 4.0}
{'user_id': 'NGTzj_44YDnPDmsD45HWeg', 'business_id': 'O8BBn8lry8lLoIFmChceGg', 'stars': 5.0}
{'user_id': 'O6wkgoJqU7KMjleSlCDGaA', 'business_id': 'EQ-TZ2eeD_E0BHuvoaeG5Q', 'stars': 5.0}


In [7]:
"""
clean_users.py

Reads the full Yelp user JSONL file, filters for users who reviewed Indianapolis restaurants
(using the cleaned review list), and writes a minimal JSONL containing only user_id and name.
"""

import json
from pathlib import Path

def clean_users():
    """
    Filters users to those present in the Indianapolis review set.

    Inputs:
    - ~/Downloads/cleaned_indianapolis_reviews.json  (filtered review file)
    - ~/Downloads/yelp_academic_dataset_user.json    (raw Yelp user data)

    Outputs:
    - ~/Downloads/cleaned_indianapolis_users.json    newline-delimited JSON with fields:
       {"user_id": "...", "name": "..."}
    - Prints the total number written and first 5 entries for verification.
    """
    # 1. Define paths
    DL       = Path.home() / "Downloads"
    INPUT    = DL / "yelp_academic_dataset_user.json"      # raw Yelp user data
    REV_FILE = DL / "cleaned_indianapolis_reviews.json"    # reviews filtered to Indy restaurants
    OUTPUT   = DL / "cleaned_indianapolis_users.json"      # final cleaned user output

    # 2. Build a set of user_ids who appear in the cleaned reviews
    user_ids = set()
    with REV_FILE.open("r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            user_ids.add(rec["user_id"].lower())

    head = []   # preview list for the first 5 users
    count = 0   # total users written

    # 3. Read raw user records, filter, and write minimal JSONL
    with INPUT.open("r", encoding="utf-8") as fin, OUTPUT.open("w", encoding="utf-8") as fout:
        for line in fin:
            rec = json.loads(line)           # parse one user record
            uid = rec["user_id"].lower()
            # 4. Only keep users in our Indianapolis set
            if uid in user_ids:
                out = {
                    "user_id": rec["user_id"],
                    "name":    rec["name"],
                }
                fout.write(json.dumps(out) + "\n")
                # 5. Collect up to 5 for preview
                if len(head) < 5:
                    head.append(out)
                count += 1

    # 6. Print summary & preview
    print(f"Wrote {count} users to {OUTPUT}\n")
    print("First 5 cleaned users:")
    for u in head:
        print(u)

if __name__ == "__main__":
    clean_users()

Wrote 77467 users to /Users/SHANAYA/Downloads/cleaned_indianapolis_users.json

First 5 cleaned users:
{'user_id': 'cxuxXkcihfCbqt5Byrup8Q', 'name': 'Rob'}
{'user_id': 'E9kcWJdJUHuTKfQurPljwA', 'name': 'Mike'}
{'user_id': 'MGPQVLsODMm9ZtYQW-g_OA', 'name': 'Jelena'}
{'user_id': 'XLs_PhrJ7Qwn_RfgMM7Djw', 'name': 'Weili'}
{'user_id': 'AkBtT43dYcttxQ3qOzPBAg', 'name': 'Sherri'}
